# Serbia Energy Optimisation

This notebook is for analysing the results of optimisation of Serbian power plants in the context of new distributed solar power generation. The problem to be solved is to decide what production each plant should produce for each hour. 

Thus, it is the *unit commitment problem*. We have 24 plants, including a hypothetical decentalised solar power plant composed of many small solar units at the domestic level. We take the demand curve of a typical day and try to match it closely with the total supply. We try to minimize:

* production cost
* emissions
* sustainability cost
* environmental cost

We have used a weighting to combine these objectives to a single objective. We have used linear programming (LP) to optimise schedules. We have used four approaches -- two grid search methods, random search, and multiobjective evolutionary search -- to explore weightings. Each leads to a Pareto front of schedules, trading-off the objectives against each other. In this notebook, we read in all the data and investigate the distributions of the above objectives, and visualise the Pareto front among them.

## Implementation

The file `run.py` contains the implementation. It reads data from several `.csv` files. We have run the optimizer many times with different weights on the above objectives, and different seeds. This notebook only carries out analysis.

First, we import all the usual Python data/visualisation libraries.

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### Problem data: plant information and demand

In [2]:
plant_info = pd.read_csv("../data/plant_info_10.csv")

In [3]:
s = plant_info.to_latex(float_format="%.3f", 
                    index=False,
                    columns=["name", "type", "production_cost", "environment_cost", "sustainability_cost", "CO2", "lower_bound", "upper_bound", "distance"],
                    header=["Name", "Type", "Prod cost", "Env cost", "Sus cost", "Emissions", "LB", "UB", "Distance"]
                   )
print(s)

\begin{tabular}{llrrrrrrr}
\toprule
Name & Type & Prod cost & Env cost & Sus cost & Emissions & LB & UB & Distance \\
\midrule
Vrla 1 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 10 & 50 & 178.830 \\
Vrla 2 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 5 & 24 & 175.470 \\
Vrla 3 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 6 & 29 & 171.320 \\
Vrla 4 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 5 & 25 & 166.940 \\
HE Pirot & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 16 & 80 & 162.140 \\
HEDJ 1 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 205 & 1026 & 152.780 \\
HEDJ 2 & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 54 & 270 & 138.400 \\
Ovčar Banja & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 1 & 6 & 58.630 \\
Međuvršje & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 1 & 7 & 54.810 \\
Uvac & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 7 & 36 & 97.830 \\
Kokin Brod & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 4 & 22 & 100.270 \\
Bistrica & hydro & 1.400 & 0.010 & 0.700 & 0.024 & 20 & 102 & 104.820 \\
Potpeć & hydro & 1.400 & 0

In [4]:
# demand.csv has one column per day; the paper's baseline is 19-Jan-21
demand = pd.read_csv("../data/demand.csv", index_col=0)[["19-Jan-21"]] \
    .rename(columns={"19-Jan-21": "demand"})

In [5]:
plt.figure(figsize=(4, 3))
sns.lineplot(x=demand.index, y=demand["demand"] / 1000, color="darkgreen", ax=plt.gca())
plt.xlabel("Hour")
plt.ylabel("Demand (MW)")
plt.tight_layout()
plt.savefig("../paper/figures/demand_curve.pdf")

The following calculates the Pareto front, given a 2D array, where each row is a point and each column is an objective.

In [6]:
# https://stackoverflow.com/questions/32791911/fast-calculation-of-pareto-front-in-python

# Fairly fast for many datapoints, less fast for many costs, somewhat readable
def is_pareto_efficient_simple(costs):
    """
    Find the pareto-efficient points
    :param costs: An (n_points, n_costs) array
    :return: A (n_points, ) boolean array, indicating whether each point is Pareto efficient
    """
    is_efficient = np.ones(costs.shape[0], dtype = bool)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient]<c, axis=1)  # Keep any point with a lower cost
            is_efficient[i] = True  # And keep self
    return is_efficient


### Reading data

The following function reads a directory of result files containing objective values into a Pandas DataFrame.

In [7]:
def read_objvals_to_pd(d, obj_names):
    results = []
    filenames = []
    for f in os.listdir(d):
        if f.startswith("objvals_"):
            df = os.path.join(d, f)
            txt = open(df).read()
            result = eval(txt)
            result = [result[name] for name in obj_names]
            result.append(f)
            result.append(d)
            results.append(result)
    results = pd.DataFrame(results)
    return results

We list the four objectives of interest.

In [8]:
obj_names = [
    "technology_cost",
    "emissions",
    "env_cost",
    "sus_cost",
]

In the next cell, we read in the `objvals` results representing the Pareto front from each run, and get all the data into a single dataframe.

In [9]:
single_pfs = []
dirs = ["grid_search", 
        "grid_search2", 
        "grid_search_norm",
        "epsilon_grid_search",
        "random_search_seed_0", 
        "random_search_seed_1", 
        "pareto_archive_seed_0",
        "pareto_archive_seed_1",
        "pareto_archive_seed_2",
        "pareto_archive_seed_3",
        "pareto_archive_seed_4"]
for dir in dirs:
    if "seed" in dir:
        algo, seed = dir.split("_seed_")
    else:
        algo = dir
        seed = 0
    for solar in (0, 10, 50, 100):
        d = f"../runs/{dir}/solar_{solar}/"
        pf = read_objvals_to_pd(d, obj_names)
        pf["solar_size"] = solar
        pf["algo"] = algo
        pf["seed"] = int(seed)
        pf.columns = ["Production cost", "Emissions", "Environmental cost", "Sustainability cost", "filename", "dirname", "New solar size (MW)", "algo", "seed"]
        pf.sort_values(by=["New solar size (MW)", "Production cost", "Emissions", "Environmental cost", "Sustainability cost"],
                       inplace=True)
        print(dir, solar, len(pf))
        single_pfs.append(pf)
results_pf = pd.concat(single_pfs)

grid_search 0 10
grid_search 10 13
grid_search 50 14
grid_search 100 15
grid_search2 0 28
grid_search2 10 36
grid_search2 50 33
grid_search2 100 26
grid_search_norm 0 10
grid_search_norm 10 15
grid_search_norm 50 14
grid_search_norm 100 19
epsilon_grid_search 0 500
epsilon_grid_search 10 500
epsilon_grid_search 50 500
epsilon_grid_search 100 500
random_search_seed_0 0 28
random_search_seed_0 10 37
random_search_seed_0 50 32
random_search_seed_0 100 35
random_search_seed_1 0 27
random_search_seed_1 10 37
random_search_seed_1 50 32
random_search_seed_1 100 33
pareto_archive_seed_0 0 27
pareto_archive_seed_0 10 37
pareto_archive_seed_0 50 37
pareto_archive_seed_0 100 38
pareto_archive_seed_1 0 29
pareto_archive_seed_1 10 37
pareto_archive_seed_1 50 36
pareto_archive_seed_1 100 39
pareto_archive_seed_2 0 28
pareto_archive_seed_2 10 37
pareto_archive_seed_2 50 37
pareto_archive_seed_2 100 36
pareto_archive_seed_3 0 31
pareto_archive_seed_3 10 39
pareto_archive_seed_3 50 38
pareto_archive_se

In [10]:
results_pf.head()

,Production cost,Emissions,Environmental cost,Sustainability cost,filename,dirname,New solar size (MW),algo,seed
6,4.858340e+08,7.488346e+07,1.369713e+08,2.147110e+08,objvals_algo_grid_search_solar_0_seed_0_3.dat,../results/grid_search/solar_0/,0,grid_search,0
1,4.858414e+08,7.479098e+07,1.368172e+08,2.145783e+08,objvals_algo_grid_search_solar_0_seed_0_8.dat,../results/grid_search/solar_0/,0,grid_search,0
5,4.858998e+08,7.472656e+07,1.367164e+08,2.144972e+08,objvals_algo_grid_search_solar_0_seed_0_7.dat,../results/grid_search/solar_0/,0,grid_search,0
3,4.860089e+08,7.463999e+07,1.365810e+08,2.143846e+08,objvals_algo_grid_search_solar_0_seed_0_4.dat,../results/grid_search/solar_0/,0,grid_search,0
2,4.862920e+08,7.449408e+07,1.363611e+08,2.142051e+08,objvals_algo_grid_search_solar_0_seed_0_5.dat,../results/grid_search/solar_0/,0,grid_search,0


### Comparing algorithms

We will take a quick look at a 2D view of the 4D Pareto front, obtained from each algorithm, for each solar scenario.

In [11]:
algo_names = {
    "grid_search": "Grid search",
    "grid_search2": "Grid search 2",
    "grid_search_norm": "Grid search norm",
    "epsilon_grid_search": "Epsilon constraint",
    "random_search": "Random search",
    "pareto_archive": "Pareto archive"
}

In [12]:
def make_comparison_plot(filename, results, xlabel, ylabel):

    if xlabel is None: xlabel = ""

    fig, ax = plt.subplots(nrows=2, ncols=len(algo_names) // 2,
                           figsize=(9.5, 6.5), 
                           sharex=True, sharey=True)
    

    last = len(algo_names) - 1

    for i, algo in enumerate(algo_names):

        row, col = i // 3, i % 3
        
        seed = 0
        
        pf = results[(results["algo"] == algo) & (results["seed"] == seed)]
        pf = pf.reset_index(drop=True)
        
        g = sns.scatterplot(
            x=xlabel,
            y=ylabel,
            data=pf, 
            s=18,
            hue="New solar size (MW)",
            palette=sns.cubehelix_palette(4, start=.5, rot=-.75, light=0.7, reverse=True),
            ax=ax[row, col],
            legend=("brief" if i == last else False),
            linewidth=0,
            alpha=0.5
        )
        
        

        ax[row, col].set_xlabel(xlabel if row == 1 else "")
        ax[row, col].set_ylabel(ylabel if col == 0 else "")

        ax[row, col].spines['top'].set_visible(False)
        ax[row, col].spines['right'].set_visible(False)

        
        if i == last:
            handles, labels = ax[row, col].get_legend_handles_labels()
            ax[row, col].legend(handles, labels,
                         title="New solar size (MW)",
                         loc="center left",
                         bbox_to_anchor=(0.6, 0.8),
                         frameon=True)

        
        ax[row, col].annotate(f"{algo_names[algo]}", xy=(0.04, 0.04), 
                       xycoords=ax[row, col].transAxes,
                      bbox=dict(boxstyle="square", fc="w"))

        
    
    plt.tight_layout()
    plt.savefig(filename)

In [13]:
make_comparison_plot("../paper/figures/compare_algorithms.pdf", results_pf, "Production cost", "Emissions")

### Pareto front progression

Next we create a plot showing how the Pareto front progresses over the generations, for a typical Pareto Archive run.

In [14]:
fronts = np.load("../runs/pareto_archive_seed_0/generations_pareto_archive_100_0.npy", allow_pickle=True)

In [15]:
def make_generations_plot(filename, fronts, xlabel=None, ylabel=None):

    fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(11.5, 3.5))

    n = len(fronts)
    idxs = (0, 1, 3, n-1)
    for i, idx in enumerate(idxs):
        
        costs = fronts[idx]
        ax[i].scatter(costs[:, 0], costs[:, 1], 
                      label=f"Gen {idx}, size {len(costs)}", s=15,
                      c=[[0.61, 0.73, 0.61]],
                      edgecolor="w",
                      lw=0.5
                     )
        ax[i].set_xlabel(xlabel if xlabel else "x")
        
        #ax[i].text(0.6, 0.8, f"Gen {idx}", transform=ax[i].transAxes)
        ax[i].legend()
        
        ax[i].spines['top'].set_visible(False)
        ax[i].spines['right'].set_visible(False)
        if i != 0:
            ax[i].set(yticklabels=[])
            #ax[i].set_xticks([])
        
        if i == 3:
            leg = ax[i].get_legend()
            #g.add_legend()
            #leg.set_bbox_to_anchor((0.6, 0.55))
            leg.draw_frame(True)        
        
    for i, idx in enumerate(idxs[:-1]):
        ax[i].set_xlim(ax[-1].get_xlim())
        ax[i].set_ylim(ax[-1].get_ylim())
        
    ax[0].set_ylabel(ylabel if ylabel else "y")
    plt.tight_layout()
    plt.savefig(filename)


In [16]:
make_generations_plot("../paper/figures/generations.pdf", fronts, "Production cost", "Emissions")

How many points in the PF, at each generation? 

In [17]:
for i, front in enumerate(fronts):
    print(i, front.shape[0])
    

0 28
1 34
2 34
3 36
4 36
5 37
6 37
7 37
8 38
9 38


### PairGrid plots for each algorithm

Next, we create a PairGrid plot which shows a scatterplot (bivariate distribution) for each pair of variables. On the diagonal we show the univariate distribution.

The next cell defines the main graph we want to make -- a PairGrid.

In [18]:
def make_pf_pairgrid(pf, algo):
    def diagfunc(x, **kws):
        ax = plt.gca()
        if kws["label"] == 0:
            ax.annotate("Best", xy=(0.1, 0.4), xycoords=ax.transAxes)
        ax.axvline(x=x.min(), ymin=0, ymax=0.3, linewidth=4, color=kws["color"])

    g = sns.PairGrid(pf, 
                     vars=["Production cost", "Emissions", "Environmental cost", "Sustainability cost"], 
                     hue="New solar size (MW)",
                     palette=sns.cubehelix_palette(4, start=.5, rot=-.75, light=0.7, reverse=True),
                     #palette=sns.color_palette("Set2"),
                     height=3, aspect=0.85
                    )
    g = g.map_diag(diagfunc)
    g = g.map_offdiag(plt.scatter, alpha=1, lw=0, s=8)
    g = g.add_legend()
    g._legend.set_bbox_to_anchor((0.25, 0.9))
    plt.tight_layout()
    plt.savefig(f"../paper/figures/pareto_front_pairgrid_{algo}.pdf")
    #plt.show()
    plt.close()


We see that a nice Pareto Front is formed between `production_cost` and `emissions`, showing that they trade-off against each other.

We also see that `emissions`, `env_cost` and `sus_cost` are closely correlated.

In [19]:

for algo in algo_names:
    seed = 0
    pf = results_pf[(results_pf["algo"] == algo) & (results_pf["seed"] == seed)]
    # print(pf)
    make_pf_pairgrid(pf, algo)

### Combining results

Up to now we have considered the algorithms separately.

Next we create a single DataFrame of combined results, and sort them by solar and then by costs, and reindex. 

In [20]:
combined = []
for solar in (0, 10, 50, 100):
    
    tmp = results_pf[results_pf["New solar size (MW)"] == solar]
    vals = tmp[["Production cost", "Emissions", "Environmental cost", "Sustainability cost"]].values
    pe = is_pareto_efficient_simple(vals)
    tmp = tmp.loc[pe, :]
    combined.append(tmp)
combined = pd.concat(combined)
combined.sort_values(by=["New solar size (MW)", "Production cost", "Emissions", "Environmental cost", "Sustainability cost"],
                   inplace=True)
combined = combined.reset_index(drop=True)  

Use that to create another PairGrid, this time for the combined results:

In [21]:
make_pf_pairgrid(combined, "combined")


### Tables of combined results

The combined front is now dominated by the epsilon-constraint sweep, which returns
one point per step (about 500 per solar size), so it is too long to tabulate in
full. Instead we write a table of how many front points each method contributes
(`paper/tables/pareto_front_counts.tex`), a short representative table per solar
size (`paper/tables/pareto_front_representative_{solar}MW.tex`), and the full
front as supplementary CSVs (`derived/pareto_front_combined_{solar}MW.csv`).

In [ ]:
# The combined front is now dominated by the epsilon-constraint sweep
# (one point per step, ~500 per solar size), so it is too long to print
# in full.  Instead we write:
#   1. a table of how many front points each method contributes;
#   2. a short representative table per solar size: points evenly spaced
#      along production cost, plus the minimum-emissions point;
#   3. the full front, one CSV per solar size, for the supplementary material.
OBJS = ["Production cost", "Emissions", "Environmental cost", "Sustainability cost"]
SOLARS = (0, 10, 50, 100)
N_REPRESENTATIVE = 10

# 1. counts
rows = []
for solar in SOLARS:
    pf = combined[combined["New solar size (MW)"] == solar]
    row = {"Solar (MW)": solar,
           "Front points": len(pf),
           "Distinct": len(pf[OBJS].round(6).drop_duplicates())}
    for algo, name in algo_names.items():
        row[name] = int((pf["algo"] == algo).sum())
    rows.append(row)
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
counts.to_csv("../derived/pareto_front_counts.csv", index=False)
with open("../paper/tables/pareto_front_counts.tex", "w") as f:
    f.write("% generated by scripts/Analyse_Results_Serbia_Solar_LP.ipynb -- do not edit\n")
    f.write(r"\begin{table}" "\n")
    f.write(r"\caption{Size of the combined Pareto front (all methods and seeds pooled, "
            r"then filtered for Pareto efficiency on all four objectives), and how many of "
            r"its points each method contributed. ``Distinct'' counts points with distinct "
            r"objective values. The epsilon-constraint sweep returns one point per step, "
            r"so its count is set by the number of steps.}" "\n")
    f.write(r"\label{tab:pareto_front_counts}" "\n")
    f.write(counts.to_latex(index=False))
    f.write(r"\end{table}" "\n")

# 2. representative points, and 3. full fronts
for solar in SOLARS:
    pf = combined[combined["New solar size (MW)"] == solar]
    pf = pf.sort_values(by=OBJS).reset_index(drop=True)

    pf.to_csv(f"../derived/pareto_front_combined_{solar}MW.csv",
              columns=OBJS + ["algo", "seed", "filename"], index_label="point")

    targets = np.linspace(pf["Production cost"].min(), pf["Production cost"].max(),
                          N_REPRESENTATIVE)
    pick = [int((pf["Production cost"] - t).abs().idxmin()) for t in targets]
    pick.append(int(pf["Emissions"].idxmin()))
    rep = pf.loc[sorted(set(pick))].copy()
    rep["Method"] = rep["algo"].map(algo_names)

    tex = rep.to_latex(float_format="%.4e", index=True,
                       columns=OBJS + ["Method"],
                       header=["Prod cost", "Emissions", "Env cost", "Sus cost", "Method"])
    caption = (rf"Representative points on the combined Pareto front, new solar size {solar}MW: "
               rf"{len(rep)} of its {len(pf)} points, evenly spaced along production cost and "
               rf"including the minimum-emissions point. The first column is the point's "
               rf"index in the full front, sorted by production cost (supplementary file "
               rf"pareto\_front\_combined\_{solar}MW.csv).")
    with open(f"../paper/tables/pareto_front_representative_{solar}MW.tex", "w") as f:
        f.write("% generated by scripts/Analyse_Results_Serbia_Solar_LP.ipynb -- do not edit\n")
        f.write(r"\begin{table}" "\n")
        f.write(r"\caption{" + caption + "}\n")
        f.write(rf"\label{{tab:pareto_front_representative_{solar}}}" "\n")
        f.write(tex)
        f.write(r"\end{table}" "\n")
    print(f"\n{solar}MW: {len(rep)} of {len(pf)} points")
    print(rep[OBJS + ["Method"]].to_string(float_format=lambda v: f"{v:.4e}"))

### Tables of best results by objective

In the following, we print out exact results for the single best point on each individual objective. We don't expect these points to be competitive when judged on multiple objectives simultaneously.

In [23]:
for solar in (0, 10, 50, 100):
    pf = combined[(combined["New solar size (MW)"] == solar)]
    pf = pf.reset_index(drop=True)
    print(f"SOLAR {solar} ######################")
    for k in ["Production cost", "Emissions", "Environmental cost", "Sustainability cost"]:
        i = pf[k].idxmin()
        print("Best", k, ", at row", i)
        print(pf.iloc[i])
        print("")

SOLAR 0 ######################
Best Production cost , at row 0
Production cost                                     485833975.913571
Emissions                                            74883464.935624
Environmental cost                                   136971309.40312
Sustainability cost                                 214710953.999196
filename               objvals_algo_grid_search_solar_0_seed_0_3.dat
dirname                              ../results/grid_search/solar_0/
New solar size (MW)                                                0
algo                                                     grid_search
seed                                                               0
Name: 0, dtype: object

Best Emissions , at row 514
Production cost                                     492647328.000012
Emissions                                            72866367.439463
Environmental cost                                  134137219.270724
Sustainability cost                                 21244

We would like to compare the best possible value on each objective, between 0MW and 100MW.

Each best value comes from a different corner solution, so as an illustration we also compare one representative compromise point per front: the knee of the production cost vs emissions trade-off, taken as the point furthest below the chord joining the min-production-cost and min-emissions corners. This choice does not depend on how the objectives are normalised. The choice of knee is debatable, so these columns illustrate a typical compromise rather than a global result.

In [24]:
def improvement_percent(x, y):
    return 100 * (x - y) / x

def knee_point(pf):
    """Knee of the production cost vs emissions trade-off: the point furthest
    below the chord joining the min-production-cost and min-emissions corners.
    The objectives are near-collinear, so this 2-D view captures the front.
    The chosen point is unchanged by rescaling either axis, so no normalisation is needed."""
    x, y = pf["Production cost"].values, pf["Emissions"].values
    a, b = pf["Production cost"].idxmin(), pf["Emissions"].idxmin()
    (xa, ya), (xb, yb) = pf.loc[a, ["Production cost", "Emissions"]], pf.loc[b, ["Production cost", "Emissions"]]
    # signed cross product: negative for points on the ideal side of the chord
    d = (xb - xa) * (y - ya) - (yb - ya) * (x - xa)
    return pf.iloc[d.argmin()]

pf0 = combined[(combined["New solar size (MW)"] == 0)]
pf100 = combined[(combined["New solar size (MW)"] == 100)]
knee0, knee100 = knee_point(pf0), knee_point(pf100)
tab = []
for k in ["Production cost", "Emissions", "Environmental cost", "Sustainability cost"]:
    v0 = pf0[k].min()
    v100 = pf100[k].min()
    tab.append((k, v0, v100, str(round(improvement_percent(v0, v100), 1)),
                knee0[k], knee100[k], str(round(improvement_percent(knee0[k], knee100[k]), 1))))
df = pd.DataFrame(tab, columns=["Objective", "best (0MW)", "best (100MW)", "improvement (%)",
                                "knee (0MW)", "knee (100MW)", "knee improvement (%)"])
print(df.to_latex(float_format="%.3e", 
                          index=False))

# write the table for the paper
SHORT = {"Production cost": "Prod cost", "Emissions": "Emissions",
         "Environmental cost": "Env cost", "Sustainability cost": "Sus cost"}
def sci(v):
    return f"{v:.3e}"
with open("../paper/tables/solar_improvement.tex", "w") as f:
    f.write("% generated by scripts/Analyse_Results_Serbia_Solar_LP.ipynb -- do not edit\n")
    f.write(r"\begin{table}" "\n")
    f.write(r"\caption{Effect of 100MW of new solar on each objective. ``Best value'' compares "
            r"the minimum of each objective on the 0MW and 100MW Pareto fronts; each minimum "
            r"comes from a different solution. ``Knee point'' compares one representative "
            r"compromise solution per front: the knee of the production cost vs emissions "
            r"trade-off, taken as the point furthest below the line joining the "
            r"minimum-production-cost and minimum-emissions solutions. The knee is an "
            r"illustration of a typical compromise, not a unique choice. Improvement is the "
            r"reduction relative to 0MW.}" "\n")
    f.write(r"\label{tab:solar_improvement}" "\n")
    f.write(r"\small\setlength{\tabcolsep}{4pt}" "\n")
    f.write(r"\begin{tabular}{lrrrrrr}" "\n")
    f.write(r"\toprule" "\n")
    f.write(r" & \multicolumn{3}{c}{Best value} & \multicolumn{3}{c}{Knee point} \\" "\n")
    f.write(r"\cmidrule(lr){2-4} \cmidrule(lr){5-7}" "\n")
    f.write(r"Objective & 0MW & 100MW & Impr. (\%) & 0MW & 100MW & Impr. (\%) \\" "\n")
    f.write(r"\midrule" "\n")
    for _, r in df.iterrows():
        f.write(" & ".join([SHORT[r["Objective"]], sci(r["best (0MW)"]), sci(r["best (100MW)"]),
                            r["improvement (%)"], sci(r["knee (0MW)"]), sci(r["knee (100MW)"]),
                            r["knee improvement (%)"]]) + r" \\" "\n")
    f.write(r"\bottomrule" "\n")
    f.write(r"\end{tabular}" "\n")
    f.write(r"\end{table}" "\n")

\begin{tabular}{lrrlrrl}
\toprule
Objective & best (0MW) & best (100MW) & improvement (%) & knee (0MW) & knee (100MW) & knee improvement (%) \\
\midrule
Production cost & 4.858e+08 & 4.818e+08 & 0.8 & 4.885e+08 & 4.878e+08 & 0.2 \\
Emissions & 7.287e+07 & 7.059e+07 & 3.1 & 7.365e+07 & 7.118e+07 & 3.4 \\
Environmental cost & 1.341e+08 & 1.299e+08 & 3.2 & 1.351e+08 & 1.306e+08 & 3.3 \\
Sustainability cost & 2.124e+08 & 2.070e+08 & 2.6 & 2.132e+08 & 2.084e+08 & 2.3 \\
\bottomrule
\end{tabular}



### What the other methods still contribute

The epsilon-constraint sweep covers the front densely, but a few points from the
other methods survive into the combined Pareto front. It is worth knowing *why*,
because the reasons are not equivalent.

The sweep is one-dimensional: it steps an upper bound on production cost and, at
each step, minimises the **sum** of the other three objectives (normalised). So a
point it fails to return falls into exactly one of three cases.

* **out-of-range** -- its production cost lies outside the swept interval. The
  sweep runs from the ideal to the nadir production cost of the payoff table, so
  this only catches a couple of extreme points per scenario.
* **resolution** -- it lies between two consecutive epsilon steps. Such a point
  differs from the front essentially along the production cost axis alone, and a
  finer sweep (larger `nsteps`) would return it. This is a tuning parameter, not
  a limitation of the method.
* **structural** -- it trades emissions, environmental cost and sustainability
  cost off *against each other*. Fusing those three into a single sum makes such
  a point unreachable at **any** resolution. These are the interesting ones: they
  quantify the residual cost of the near-collinearity assumption that justifies
  the one-dimensional sweep in the first place.

In [25]:
# The four objective columns, and a helper to split one solar scenario's
# combined front into the epsilon-constraint points and the rest.
obj_cols = ["Production cost", "Emissions", "Environmental cost", "Sustainability cost"]

def split_front(solar):
    pf = combined[combined["New solar size (MW)"] == solar]
    eps = pf[pf["algo"] == "epsilon_grid_search"].sort_values("Production cost")
    oth = pf[pf["algo"] != "epsilon_grid_search"]
    return pf, eps, oth

# Classify every non-epsilon point on the combined front into one of three
# causes.  The epsilon sweep is one-dimensional: it steps the bound on
# production cost and, at each step, minimises the *sum* of the other three
# (normalised).  So it can fail to return a point for exactly three reasons.
#
#   out-of-range  the point's production cost lies outside the swept interval
#   resolution    the point falls between two consecutive epsilon steps; it
#                 differs from the front only along the production cost axis,
#                 and a finer sweep (larger nsteps) would return it
#   structural    the point trades emissions / environmental / sustainability
#                 cost off *against each other*.  Summing those three into one
#                 scalar makes such a point unreachable at ANY resolution.
def classify(solar, tol=1e-6):
    pf, eps, oth = split_front(solar)
    ev = eps[obj_cols].values
    pc = ev[:, 0]
    out = []
    for i, r in oth.iterrows():
        v = r[obj_cols].values.astype(float)
        if v[0] < pc[0] or v[0] > pc[-1]:
            out.append((i, r["algo"], "out-of-range", np.nan, np.nan))
            continue
        # does this point trade off *within* the three summed objectives,
        # against any epsilon point?  better on one of them, worse on another.
        d = ev[:, 1:] - v[1:]
        t = tol * np.abs(v[1:])
        mixed = (d < -t).any(axis=1) & (d > t).any(axis=1)
        # distance to the nearest epsilon point, normalised per objective
        rng = pf[obj_cols].values.ptp(axis=0)
        near = np.linalg.norm((v - ev) / rng, axis=1).min()
        if mixed.any():
            j = np.where(mixed)[0][np.abs(d[mixed] / np.abs(v[1:])).max(axis=1).argmax()]
            out.append((i, r["algo"], "structural", near, eps.index.values[j]))
        else:
            out.append((i, r["algo"], "resolution", near, np.nan))
    return pd.DataFrame(out, columns=["idx", "algo", "cause",
                                      "dist to nearest eps pt", "eps partner"]).set_index("idx")

for solar in (0, 10, 50, 100):
    cls = classify(solar)
    print(f"=== solar {solar}MW: {len(cls)} non-epsilon points on the combined front")
    print(pd.crosstab(cls["algo"], cls["cause"]))
    print()

=== solar 0MW: 36 non-epsilon points on the combined front
cause             out-of-range  resolution  structural
algo                                                  
grid_search                  2           7           1
grid_search2                 0          19           0
grid_search_norm             1           2           0
pareto_archive               0           1           0
random_search                0           3           0

=== solar 10MW: 53 non-epsilon points on the combined front
cause             out-of-range  resolution  structural
algo                                                  
grid_search                  2           9           2
grid_search2                 0          23           1
grid_search_norm             0           5           2
pareto_archive               0           1           3
random_search                0           4           1

=== solar 50MW: 43 non-epsilon points on the combined front
cause             out-of-range  resolution  struc

In [26]:
# The "structural" points in detail.  For each one we show the epsilon point it
# trades off against: the difference is a genuine re-allocation between
# emissions / environmental cost / sustainability cost at essentially the same
# production cost, so one of the three goes down while another goes up.
def structural_examples(solar, n=5):
    pf, eps, oth = split_front(solar)
    cls = classify(solar)
    rows = []
    for i, r in cls[cls["cause"] == "structural"].iterrows():
        v = combined.loc[i, obj_cols].values.astype(float)
        w = combined.loc[r["eps partner"], obj_cols].values.astype(float)
        rel = 100 * (v - w) / np.abs(w)
        rows.append([i, r["algo"], int(r["eps partner"])] + list(rel))
    df = pd.DataFrame(rows, columns=["idx", "algo", "eps partner"] +
                      [f"{c} (% vs eps)" for c in obj_cols])
    # rank by how large the within-three trade-off is
    key = df[[f"{c} (% vs eps)" for c in obj_cols[1:]]].abs().max(axis=1)
    return df.assign(_k=key).sort_values("_k", ascending=False).drop(columns="_k").head(n)

for solar in (0, 10, 50, 100):
    print(f"=== solar {solar}MW: structural points (a trade-off the summed epsilon objective cannot express)")
    print(structural_examples(solar).to_string(index=False,
          float_format=lambda x: f"{x:+.4f}"))
    print()

=== solar 0MW: structural points (a trade-off the summed epsilon objective cannot express)
 idx        algo  eps partner  Production cost (% vs eps)  Emissions (% vs eps)  Environmental cost (% vs eps)  Sustainability cost (% vs eps)
 514 grid_search          534                     -0.0568               -0.0004                        +0.0004                         +0.0154

=== solar 10MW: structural points (a trade-off the summed epsilon objective cannot express)
 idx             algo  eps partner  Production cost (% vs eps)  Emissions (% vs eps)  Environmental cost (% vs eps)  Sustainability cost (% vs eps)
1035      grid_search         1086                     -0.1414               -0.0003                        +0.0012                         +0.0483
1038 grid_search_norm         1086                     -0.1365               -0.0003                        +0.0012                         +0.0463
1039     grid_search2         1086                     -0.1365               -0.0003  

In [38]:
# Where on the front the other methods still contribute.  The epsilon front is
# drawn as a line; non-epsilon points are marked by the reason they survive.
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(13, 3.4), sharey=False)
style = {"resolution": ("o", "tab:blue"),
         "structural": ("D", "tab:red"),
         "out-of-range": ("s", "tab:green")}
for a, solar in zip(ax, (0, 10, 50, 100)):
    pf, eps, oth = split_front(solar)
    cls = classify(solar)
    a.plot(eps["Production cost"] / 1e8, eps["Emissions"] / 1e7,
           "-", color="0.6", lw=1, zorder=1, label="Epsilon front")
    for cause, (m, c) in style.items():
        sel = cls[cls["cause"] == cause].index
        if len(sel) == 0:
            continue
        a.scatter(combined.loc[sel, "Production cost"] / 1e8,
                  combined.loc[sel, "Emissions"] / 1e7,
                  marker=m, s=26, c=c, edgecolor="k", linewidth=0.4,
                  zorder=3, label=f"{cause} ({len(sel)})")
    a.set_title(f"{solar}MW")
    a.set_xlabel("Production cost ($\\times 10^8$)")
    a.legend(fontsize=6, loc="upper right")
ax[0].set_ylabel("Emissions ($\\times 10^7$)")
plt.tight_layout()
plt.savefig("../paper/figures/eps_vs_others.pdf")

In [39]:
# The same layout again, but with the surviving points coloured by which
# algorithm contributed them, rather than by why epsilon missed them.
contrib_style = {
    "grid_search":      ("o", "tab:blue"),
    "grid_search2":     ("s", "tab:orange"),
    "grid_search_norm": ("^", "tab:green"),
    "random_search":    ("v", "tab:purple"),
    "pareto_archive":   ("D", "tab:red"),
}

fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(13, 3.4))
for a, solar in zip(ax, (0, 10, 50, 100)):
    pf, eps, oth = split_front(solar)
    a.plot(eps["Production cost"] / 1e8, eps["Emissions"] / 1e7,
           "-", color="0.6", lw=1, zorder=1,
           label=f"{algo_names['epsilon_grid_search']} ({len(eps)})")
    for algo, (m, c) in contrib_style.items():
        sel = oth[oth["algo"] == algo]
        if len(sel) == 0:
            continue
        a.scatter(sel["Production cost"] / 1e8, sel["Emissions"] / 1e7,
                  marker=m, s=26, c=c, edgecolor="k", linewidth=0.4,
                  zorder=3, label=f"{algo_names[algo]} ({len(sel)})")
    a.set_title(f"{solar}MW")
    a.set_xlabel("Production cost ($\\times 10^8$)")
    a.legend(fontsize=6, loc="upper right")
ax[0].set_ylabel("Emissions ($\\times 10^7$)")
plt.tight_layout()
plt.savefig("../paper/figures/eps_vs_others_by_algo.pdf")

### Visualising schedules

Next we are going to plot some schedules as heatmaps.

First we find the two schedules of min Env cost (for 0MW and 100MW), and then same for min Production cost - these are is to demonstrate how the extra 100MW changes things.

In [28]:
solar = 0
pf = combined[(combined["New solar size (MW)"] == solar)]
pf[
    (pf["Environmental cost"] == pf["Environmental cost"].min())
  ]



,Production cost,Emissions,Environmental cost,Sustainability cost,filename,dirname,New solar size (MW),algo,seed
535,4.929274e+08,7.286668e+07,1.341367e+08,2.124108e+08,objvals_algo_grid_search_solar_0_seed_0_1.dat,../results/grid_search/solar_0/,0,grid_search,0


In [29]:
solar = 100 # 100MW
pf = combined[(combined["New solar size (MW)"] == solar)]
pf[
    (pf["Environmental cost"] == pf["Environmental cost"].min())
  ]

,Production cost,Emissions,Environmental cost,Sustainability cost,filename,dirname,New solar size (MW),algo,seed
2174,4.963123e+08,7.059351e+07,1.298906e+08,2.069894e+08,objvals_algo_grid_search_solar_100_seed_0_1.dat,../results/grid_search/solar_100/,100,grid_search,0


In [30]:
solar = 0
pf = combined[(combined["New solar size (MW)"] == solar)]
pf[
    (pf["Production cost"] == pf["Production cost"].min())
  ]

,Production cost,Emissions,Environmental cost,Sustainability cost,filename,dirname,New solar size (MW),algo,seed
0,4.858340e+08,7.488346e+07,1.369713e+08,2.147110e+08,objvals_algo_grid_search_solar_0_seed_0_3.dat,../results/grid_search/solar_0/,0,grid_search,0


In [31]:
solar = 100 # 100MW
pf = combined[(combined["New solar size (MW)"] == solar)]
pf[
    (pf["Production cost"] == pf["Production cost"].min())
  ]

,Production cost,Emissions,Environmental cost,Sustainability cost,filename,dirname,New solar size (MW),algo,seed
1630,4.817631e+08,7.328562e+07,1.338606e+08,2.115401e+08,objvals_algo_grid_search_solar_100_seed_0_4.dat,../results/grid_search/solar_100/,100,grid_search,0


### Schedules along the front

Previously we compared two schedules on opposite sides of an apparent gap in the
100MW front, to show a step change in strategy. The epsilon-constraint results
show that gap was a sampling artifact -- the front is continuous -- so there is
no longer a discontinuity to straddle.

The replacement analysis uses what the epsilon sweep gives us that the weight
searches could not: points at a known, uniform spacing along the front, so
"position along the front" is a meaningful axis. We can therefore ask how the
schedule *evolves* as we trade production cost against the other objectives,
rather than comparing two isolated points.

Two things come out of it. The migration is continuous and largely monotone --
solar displaces coal steadily as we allow production cost to rise. And the rate
of reorganisation is not uniform: it spikes at particular production costs, which
are the vertices of the piecewise-linear front, where the set of marginal plants
changes. Those spikes are the honest version of the old "step change in
strategy" -- a change of basis rather than a gap.

In [32]:
# Load every schedule along the epsilon front for one scenario, as a tensor
# (front position, plant, hour).  The epsilon sweep gives us points at a known,
# uniform spacing in production cost, so "front position" is a meaningful axis.
def front_schedules(solar):
    pf, eps, oth = split_front(solar)
    S = []
    for _, r in eps.iterrows():
        f = r["filename"].replace("objvals", "relative_supply").replace(".dat", ".csv")
        S.append(np.genfromtxt(os.path.join(r["dirname"], f)))
    names = pd.read_csv(f"../data/plant_info_{solar}.csv")["name"].values
    return eps, np.array(S), names

solar = 100
eps100, S, names = front_schedules(solar)
util = S.mean(axis=2)   # mean relative output per plant, per front point
print("schedules:", S.shape, " (front points, plants, hours)")

schedules: (499, 24, 24)  (front points, plants, hours)


In [33]:
# Which plants move as we walk along the front, and in which direction?
swing = util.max(axis=0) - util.min(axis=0)
trend = np.array([np.corrcoef(np.arange(len(util)), util[:, k])[0, 1]
                  if swing[k] > 1e-9 else 0.0 for k in range(util.shape[1])])
tab = pd.DataFrame({"plant": names, "at min prod cost": util[0],
                    "at max prod cost": util[-1], "swing": swing,
                    "trend (corr)": trend})
tab = tab.sort_values("swing", ascending=False)
print(tab.head(10).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

            plant  at min prod cost  at max prod cost  swing  trend (corr)
Solarna Elektrana             0.167             0.961  0.794         0.825
        New Solar             0.305             1.000  0.695         0.954
          TETO NS             0.236             0.408  0.172         0.962
          TENT B1             0.358             0.216  0.142        -0.945
           Vrla 1             0.568             0.675  0.132         0.278
           Vrla 2             0.604             0.703  0.129         0.278
           Vrla 3             0.624             0.703  0.112         0.189
           HEDJ 1             0.835             0.859  0.067         0.444
           Vrla 4             0.700             0.700  0.033         0.115
         HE Pirot             0.700             0.700  0.033        -0.118


In [34]:
# The same information as a picture: plant utilisation against position along
# the front.  Continuous migration, not a step change.
top = np.argsort(-swing)[:6]
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 3.8))
pcv = eps100["Production cost"].values / 1e8
for k in top:
    ax[0].plot(pcv, util[:, k], lw=1.5, label=names[k])
ax[0].set_xlabel("Production cost ($\\times 10^8$)")
ax[0].set_ylabel("Mean relative output")
ax[0].legend(fontsize=7)
ax[0].set_title(f"Most-changed plants along the front, {solar}MW")

order = np.argsort(-swing)
im = ax[1].imshow(util[:, order].T, aspect="auto", cmap="Reds", vmin=0, vmax=1,
                  extent=[pcv[0], pcv[-1], len(order), 0])
ax[1].set_yticks(np.arange(len(order)) + 0.5)
ax[1].set_yticklabels(names[order], fontsize=6)
ax[1].set_xlabel("Production cost ($\\times 10^8$)")
ax[1].set_title("All plants (sorted by swing)")
fig.colorbar(im, ax=ax[1], label="Mean relative output")
plt.tight_layout()
plt.savefig(f"../paper/figures/schedule_trend_{solar}MW.pdf")

In [35]:
# The left panel again, on its own and without a title, for use as a figure
# with its own caption.
plt.figure(figsize=(5.5, 3.8))
for k in np.argsort(-swing)[:6]:
    plt.plot(eps100["Production cost"].values / 1e8, util[:, k], lw=1.5, label=names[k])
plt.xlabel("Production cost ($\\times 10^8$)")
plt.ylabel("Mean relative output")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"../paper/figures/schedule_trend_lines_{solar}MW.pdf")

In [36]:
# The old analysis picked two schedules on opposite sides of an apparent gap in
# the front.  The gap turned out to be a sampling artifact, so there is no
# longer a discontinuity to straddle.  Instead we locate the largest
# *structural* switch: the adjacent pair of front points whose schedules differ
# most.  The front is continuous in objective space, but the schedule still
# reorganises faster at some places than others -- these are the vertices of
# the piecewise-linear front, where the set of marginal plants changes.
step = np.abs(np.diff(util, axis=0)).sum(axis=1)
print(f"schedule change between adjacent front points: "
      f"median {np.median(step):.4f}, max {step.max():.4f}")

# The two ends of the front reorganise most, but that is a boundary effect;
# the interesting switches are interior, so we ignore the first and last few.
# as a fraction of the front, so this is unchanged by nsteps
margin = max(5, len(step) // 20)
interior = step.copy()
interior[:margin] = -1
interior[-margin:] = -1
i = int(interior.argmax())
lo, hi = eps100.index[i], eps100.index[i + 1]
print(f"\nlargest interior switch at front step {i} -> {i+1} "
      f"(combined idx {lo} -> {hi}), L1 change {step[i]:.4f} "
      f"= {step[i]/np.median(step):.1f}x the median step")
print(combined.loc[[lo, hi], obj_cols].to_string(float_format="%.6e"))
print("\nplants moving most across that switch:")
delta = util[i + 1] - util[i]
for k in np.argsort(-np.abs(delta))[:6]:
    print(f"   {names[k]:26s} {util[i, k]:.3f} -> {util[i+1, k]:.3f}  ({delta[k]:+.3f})")

plt.figure(figsize=(5.5, 3))
plt.plot(eps100["Production cost"].values[:-1] / 1e8, step, lw=1)
plt.axvline(eps100["Production cost"].values[i] / 1e8, color="r", ls="--", lw=1,
            label=f"largest interior switch (step {i})")
plt.xlabel("Production cost ($\\times 10^8$)")
plt.ylabel("Schedule change (L1)")
plt.title(f"Rate of schedule reorganisation, {solar}MW")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"../paper/figures/schedule_switch_rate_{solar}MW.pdf")

# these two indices replace the old hardcoded LHS/RHS-of-gap pair
switch_lo, switch_hi = lo, hi

schedule change between adjacent front points: median 0.0028, max 0.1072

largest interior switch at front step 488 -> 489 (combined idx 2162 -> 2163), L1 change 0.0446 = 15.7x the median step
      Production cost    Emissions  Environmental cost  Sustainability cost
2162     4.959916e+08 7.059333e+07        1.298913e+08         2.070273e+08
2163     4.960208e+08 7.059324e+07        1.298912e+08         2.070236e+08

plants moving most across that switch:
   Solarna Elektrana          0.570 -> 0.609  (+0.040)
   Vrla 1                     0.682 -> 0.677  (-0.005)
   TETO NS                    0.408 -> 0.408  (+0.000)
   TENT B2                    1.000 -> 1.000  (+0.000)
   TENT B1                    0.216 -> 0.216  (+0.000)
   TENT A                     1.000 -> 1.000  (+0.000)


In [37]:
# Schedules to plot as heatmaps.  The indices are looked up rather than
# hardcoded, so this cell stays correct if the results are regenerated.
# The last two replace the old LHS/RHS-of-gap pair with the two sides of the
# largest interior schedule switch, found in the cell above.
def idx_of_min(solar, col):
    pf = combined[combined["New solar size (MW)"] == solar]
    return pf[col].idxmin()

to_do = [(0,   idx_of_min(0, "Environmental cost"),   "min_env_cost_0MW"),
         (0,   idx_of_min(0, "Production cost"),      "min_prod_cost_0MW"),
         (100, idx_of_min(100, "Environmental cost"), "min_env_cost_100MW"),
         (100, idx_of_min(100, "Production cost"),    "min_prod_cost_100MW"),
         (100, switch_lo, "before_switch_100MW"),
         (100, switch_hi, "after_switch_100MW")]

for solar, idx, title in to_do:
    pf = combined[(combined["New solar size (MW)"] == solar)]
    f = pf.loc[idx]["filename"]
    d = pf.loc[idx]["dirname"]
    algo = pf.loc[idx]["algo"]
    f = f.replace("objvals", "relative_supply").replace(".dat", ".csv")

    print(title, "| idx", idx, "| algo", algo)
    print(f)
    plt.figure(figsize=(5,5))
    xbest = np.genfromtxt(os.path.join(d, f), delimiter=" ")
    pi = pd.read_csv(f"../data/plant_info_{solar}.csv")
    heatmap = sns.heatmap(xbest, cmap="Reds", linewidths=0.1, linecolor="k", vmin=0, vmax=1);
    heatmap.set_xlabel("Hour")
    heatmap.set_yticklabels(pi["name"], rotation=0);
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"../paper/figures/schedule_{algo}_{solar}_{idx}_{title}.pdf")


min_env_cost_0MW | idx 535 | algo grid_search
relative_supply_algo_grid_search_solar_0_seed_0_1.csv
min_prod_cost_0MW | idx 0 | algo grid_search
relative_supply_algo_grid_search_solar_0_seed_0_3.csv
min_env_cost_100MW | idx 2174 | algo grid_search
relative_supply_algo_grid_search_solar_100_seed_0_1.csv
min_prod_cost_100MW | idx 1630 | algo grid_search
relative_supply_algo_grid_search_solar_100_seed_0_4.csv
before_switch_100MW | idx 2162 | algo epsilon_grid_search
relative_supply_algo_epsilon_grid_search_solar_100_seed_0_488.csv
after_switch_100MW | idx 2163 | algo epsilon_grid_search
relative_supply_algo_epsilon_grid_search_solar_100_seed_0_489.csv


### Sensitivity: how much cheaper would solar have to be?

`../src/sensitivity.py` asks, for each technology, how far its production cost
must move before the cost-optimal schedule changes *at all*.  At 100MW solar's
first flip is at about $-9\%$ -- but it is worth almost nothing, moving about
250 MWh on a 138 GWh day (52 MWh of it onto solar), and is invisible in a
schedule heatmap.  The schedule only reorganises visibly once solar runs around
the clock, which takes about $-73\%$.

The two schedules below are that contrast: both are 100MW new solar at
*minimum production cost*, differing only in the price of solar.  They are a
matched pair in their own right, and deliberately separate from the
0MW/100MW pair above, which is at minimum environmental cost -- price and
capacity are different levers and are not comparable across those figures.
The size of the number needed is itself the result: 100MW of solar is small
against a 4.8GW system, so price has to move a long way before dispatch
notices.

In [ ]:
# A matched pair of standalone schedules: 100MW new solar at minimum
# production cost, with solar priced as given and at a quarter of that.  Both
# are written by ../src/sensitivity.py (run it first) from the same solve
# path, so the only difference between the two figures is the solar price --
# the baseline panel is checked there against the stored min-production-cost
# point from the epsilon sweep.  One figure per file, with the same shape and
# furniture as the schedule cell above, so all of them crop identically.
pi100 = pd.read_csv("../data/plant_info_100.csv")
to_do = [("schedule_baseline_100MW.csv",    "min_prod_cost_100MW"),
         ("schedule_solar-75pct_100MW.csv", "min_prod_cost_100MW_solar-75pct")]

for fn, title in to_do:
    rel = np.genfromtxt(os.path.join("../derived", fn))
    solar = rel[(pi100["type"] == "solar").values]
    print(f"{title:34s} solar runs in "
          f"{int((solar.sum(axis=0) > 1e-6).sum()):2d} of 24 hours")

    plt.figure(figsize=(5,5))
    heatmap = sns.heatmap(rel, cmap="Reds", linewidths=0.1, linecolor="k", vmin=0, vmax=1);
    heatmap.set_xlabel("Hour")
    heatmap.set_yticklabels(pi100["name"], rotation=0);
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"../paper/figures/schedule_{title}.pdf")